# Phase 4 (Court Judgment Prediction, CJPE) — Run #4 on Phase-1 **Decoder-only model 03** (ALiBi + Multi-Head Latent Attention), with **rsLoRA**

Config-driven like the other Phase-4 notebooks: edit **Cell 2** only. This is the **fourth** Phase-4 run of the plan: the **full-size** Dec-03 (13 blocks, 53.9M parameters), unmasked through symmetric ALiBi, adapted with **rank-stabilised LoRA (rsLoRA)** at a high rank. Data loading, chunking, the document model, metrics, threshold tuning, the submission writer and the run log are *identical* to the Dec-02, Dec-01 and Enc-Dec notebooks, so all runs are comparable. It is also the full-size sibling of the KD student (a 7-block cut of this same model).

### The task in one paragraph
Each case is one long judgment (`{"id", "text"}`; train/dev also have a binary label). Predict **0 or 1** per case. Metric: **macro-F1**. The test file must give **exactly one integer prediction per test id**.

### Model = Phase-1 backbone per chunk → mean-pool → aggregator over chunks → `Linear(512 → 2)`
1. **Select** at most `MAX_CHUNKS` chunks of `CHUNK_TOKENS` tokens from the case (`DOC_VIEW`: `tail` / `head` / `head_tail` / `spread`).
2. **Encode** every chunk with the (adapted) backbone and **mean-pool** its tokens into `h_c`.
3. **Aggregate** the chunk vectors (`AGG`: `mean`, `attn` = `α_c = softmax(vᵀ tanh(W h_c))`, or `hier` = a small Transformer over the chunk vectors).
4. `dropout → Linear(512 → 2)`, class-weighted cross-entropy, decision threshold tuned on dev.

Cost: attention over a whole case of `K·L` tokens would be `O((KL)²)`; independent chunks cost `O(K·L²)`.

### What this architecture offers
| | Decoder 01 | Decoder 02 | **Decoder 03 (this notebook)** |
|---|---|---|---|
| positions | none (NoPE) | RoPE | **ALiBi**: a fixed bias `−m_h·(i−j)` on the attention logits, no position parameters |
| attention | MHA | local/global + GQA | **MLA**: K and V of every head are rebuilt from a 128-dim latent `c_t` |
| layers | 12 | 12 | **13** |
| can it be unmasked? | no | yes | **yes**: the distance lives in the *bias*, so the symmetric version `−m_h·|i−j|` is well defined |

### rsLoRA (Kalajdzievski 2023) — why the scaling factor matters
LoRA adds `γ · B·A·x` to a frozen layer, with `A ∈ R^{r×d_in}` (Kaiming-uniform, entries of variance ≈ 1/(3·d_in)) and `B = 0` at the start. Standard LoRA uses `γ = α/r`. Take **one SGD step** on `B` with a linear loss `L = g·h`: `∇_B L = γ·g·(A x)ᵀ`, so `B₁ = −η·γ·g·(A x)ᵀ` and the change of the layer output is

`Δh = γ·B₁·A·x = −η·γ²·‖A x‖²·g`,  with  `‖A x‖² = Σᵢ (aᵢ·x)² ≈ r·‖x‖²/(3·d_in)`  (it grows **linearly in r**).

So `|Δh| ∝ γ²·r`. With `γ = α/r` this is `∝ α²/r`: **the larger the rank, the smaller the update**, which is why plain LoRA at high rank learns no better than at low rank (the extra capacity is never used). The unique choice that keeps `γ²·r` constant, i.e. makes learning *stable and rank-independent*, is **`γ = α/√r`**. That is the whole of rsLoRA: a single change of the scaling constant, no extra parameters. The notebook prints a numerical check of this (`SCALING_DEMO`, Cell 17).

Which adapters: every `Linear` named in `ADAPTER_TARGETS` (MLA: `q_proj`, `kv_down`, `k_up`, `v_up`, `o_proj`; SwiGLU: `gate_proj`, `up_proj`, `down_proj`). At the default rank 64: about 8.1M trainable parameters (15 % of the model) against 2.0M for LoRA at rank 16.

### `METHOD`
| `METHOD` | What is trained |
|---|---|
| `"rslora"` | rsLoRA matrices (`γ = α/√r`, default r = 64, α = 16) + aggregator/head (**default**) |
| `"lora"` | standard LoRA (`γ = α/r`, default r = 16, α = 32) + aggregator/head |
| `"partial"` | the top `PARTIAL_K` blocks + final norm + aggregator/head (cheap middle ground, run with `ATTN_MODE="bidir_top"`) |
| `"full"` | everything (full-fine-tuning reference) |
| `"norm_only"` | all RMSNorm gains (incl. MLA's latent norm) + aggregator/head |

### `ATTN_MODE`
`"bidir"` (default): all 13 blocks use symmetric ALiBi and no causal mask, so every token sees its whole chunk. `"bidir_top"`: the lower blocks stay causal (exactly the Phase-1 function), the top `BIDIR_TOP_K` blocks are unmasked; the natural partner of `partial`, where the frozen lower blocks must not see an attention pattern they were never trained for. `"causal"`: the Phase-1 mask in every block (control: does unmasking help?).

### What this notebook writes
`submission.zip` (verified like the evaluator would), `run_log.json` (parameters, peak GPU memory, train time, inference ms per case, dev metrics, per-class P/R/F1, confusion matrix, per-slot attention profile, FP/FN by length bucket), `dev_errors.json`, and (if `EXPORT_SOFT_LABELS`) the soft labels for KD.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p4_dec03_rslora"
METHOD     = "rslora"         # "rslora" | "lora" | "partial" | "full" | "norm_only"
SMOKE_TEST = False            # True: ~150 steps on a small random subset, dev check only, NO submission written
ATTN_MODE  = "bidir"          # "bidir" | "bidir_top" | "causal"   (see the first markdown cell)
PARTIAL_K   = 4               # METHOD="partial": train the top K blocks (0 = final norm + head only, a linear probe)
BIDIR_TOP_K = 4               # ATTN_MODE="bidir_top": the top K blocks become bidirectional (keep equal to PARTIAL_K for partial)
SCALING_DEMO = True           # print the one-step numerical check of the LoRA vs rsLoRA scaling (Cell 17), takes a second

# ---- Phase-1 checkpoint (written by caps_decoder_only_architecture_03.ipynb: runs/<ARCH>/best.pt) ---------------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-03-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_4_CJPE/{Train/train.json, Train/dev.json, Test/test.json}
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_4_CJPE/ (checked first)
FINAL_FIT  = False                                # True: train on train+dev, NO validation (use after the dev run, EPOCHS = best epoch)

# ---- Long-document representation -----------------------------------------------------------
DOC_VIEW     = "tail"        # "tail" | "head" | "head_tail" | "spread"   (see the first markdown cell)
CHUNK_TOKENS = 512           # tokens per chunk INCLUDING the leading [BOS]  (<= 1024, the Phase-1 context)
MAX_CHUNKS   = 4             # chunks per case  => at most MAX_CHUNKS*(CHUNK_TOKENS-1) tokens are read
AGG          = "attn"        # "mean" | "attn" | "hier"
HIER_LAYERS  = 1             # AGG="hier": number of Transformer layers over the chunk vectors
CHAR_PER_TOK = 8             # tokenisation shortcut for long texts: only a slice of CHAR_PER_TOK chars per wanted token is tokenised
N_PROC       = 4             # worker processes for tokenisation (1 = serial)

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 4
BATCH_DOCS   = 4             # cases per micro-batch (each case = up to MAX_CHUNKS sequences)
ACCUM        = 4             # gradient accumulation  => effective batch = BATCH_DOCS*ACCUM cases
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
GRAD_CKPT    = False         # True: recompute block activations in the backward pass (less memory, ~30% slower)
TIME_BUDGET_MIN = None       # e.g. 150: stop training after this many minutes (the best epoch so far is kept)
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "rslora":    dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=64, alpha=16, dropout=0.05),     # gamma = alpha / sqrt(r) = 2.0
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),     # gamma = alpha / r      = 2.0
    "partial":   dict(lr=2e-4, head_lr=1e-3, wd=0.01),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
# every Linear name in this architecture: MHA/GQA-style (q,k,v,o), MLA (q, kv_down, k_up, v_up, o) and the SwiGLU FFN
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "kv_down", "k_up", "v_up", "gate_proj", "up_proj", "down_proj")
HEAD_DROPOUT    = 0.1
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
LABEL_SMOOTHING = 0.0

# ---- Decision threshold tuned on dev (macro-F1 post-processing) -----------------------------
THRESH_TUNE     = True
THRESH_MIN_GAIN = 0.005                           # keep the tuned threshold only if a 2-fold held-out check on dev gains this much macro-F1
THRESH_FINAL    = None                            # used when FINAL_FIT=True: paste the bias printed by the dev run (or None)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 800, 200

# ---- Knowledge distillation (run #5 of the Phase-4 plan): optional teacher (~62M with the adapters, larger than the 7-block student) --
EXPORT_SOFT_LABELS = False        # True: save document-level log-probabilities of train/dev (a valid teacher for the 36.6M student if it is your best run)

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE in ("causal", "bidir", "bidir_top")
assert DOC_VIEW in ("tail", "head", "head_tail", "spread") and AGG in ("mean", "attn", "hier")
assert 16 <= CHUNK_TOKENS <= 1024 and MAX_CHUNKS >= 1, "chunks longer than 1024 exceed the Phase-1 RoPE table"
BODY = CHUNK_TOKENS - 1                           # tokens of text per chunk (the first position is [BOS])
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | ATTN_MODE={ATTN_MODE} | view={DOC_VIEW} {MAX_CHUNKS}x{CHUNK_TOKENS} | agg={AGG} | "
      f"batch={BATCH_DOCS}x{ACCUM} | epochs={EPOCHS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

RUN p4_dec03_rslora | METHOD=rslora | ATTN_MODE=bidir | view=tail 4x512 | agg=attn | batch=4x4 | epochs=4 | final_fit=False | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# Sentences are tokenised exactly as given (no clean_text / normalisation).
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_03.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 13,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "alibi",
  "rope_theta": 10000.0,
  "mixer_type": "mla",
  "kv_latent_dim": 128,
  "mla_latent_norm": true,
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [8]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [10]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [12]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [13]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} mixer={lm_cfg.mixer_type} (latent {lm_cfg.kv_latent_dim}) | "
      f"layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} heads={lm_cfg.n_head} d_ff={lm_cfg.d_ff} | "
      f"params {pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE in ("bidir", "bidir_top") and lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"ATTN_MODE='{ATTN_MODE}' is implemented for ALiBi checkpoints (pos_type='alibi'); this one has "
                       f"pos_type='{lm_cfg.pos_type}'. Use ATTN_MODE='causal'.")
if METHOD == "partial":
    assert 0 <= PARTIAL_K <= lm_cfg.n_layer, "PARTIAL_K must be between 0 and the number of layers"
if lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"this notebook handles the ALiBi checkpoints of Decoder-03 (alibi_mla / alibi_mha); got pos_type='{lm_cfg.pos_type}'. "
                       f"For a RoPE or NoPE checkpoint use the Dec-02 / Dec-01 notebooks.")

Phase-1 model | pos_type=alibi mixer=mla (latent 128) | layers=13 d_model=512 heads=8 d_ff=1408 | params 53.89M (embedding 16.38M)
checkpoint extra: step 4591 | val_loss 1.8845605307433289
LM loss on a legal sentence: 1.947  (ln V = 10.37)


## Phase-4 data: cases, labels, statistics
Cases are `{"id", "text"}` (+ `"label"` for train/dev). The loader accepts a few equivalent layouts (a list, or a dict keyed by id; `text` may be a string or a list of strings; the label key may be `label`/`labels`/`y`/`judgment`/`outcome`), and **fails loudly** if a label is not 0/1. The statistics cell tells you how much of a typical case your `MAX_CHUNKS × CHUNK_TOKENS` budget covers. If the schema printed below is not what you expect, paste it to Claude.

In [14]:
# Cell 13 — Phase-4 data: train / dev / test cases
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P4 = "Phase_4_CJPE"

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_4_CJPE folder and set LOCAL_DATA_DIR.") from ex

def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default

def parse_cases(raw, split, has_labels):
    """-> list of {uid, id, text, y|None, nchar}.  uid = '<split>::<position in the file>' (stable, used internally and by the KD export)."""
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"text": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ("id", "case_id", "doc_id", "document_id"), i)
        text = _first(e, ("text", "case_text", "document", "judgment_text", "content"))
        if isinstance(text, list): text = "\n".join(str(t) for t in text)
        if not isinstance(text, str):
            raise ValueError(f"{split} case {rid}: cannot find the case text. Keys: {list(e)}")
        y = None
        if has_labels:
            lab = _first(e, ("label", "labels", "y", "judgment", "outcome", "decision"))
            try: y = int(lab)
            except Exception: raise ValueError(f"{split} case {rid}: label {lab!r} is not 0/1. Keys: {list(e)}")
            if y not in (0, 1): raise ValueError(f"{split} case {rid}: label {y} is not 0/1")
        docs.append(dict(uid=f"{split}::{i}", id=rid, text=text, y=y, nchar=len(text)))
    return docs

def load_split(rel, split, has_labels):
    raw = json.loads(fetch(rel).read_text(encoding="utf-8"))
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_cases(raw, split, has_labels)

TRAIN_ALL = load_split(f"{P4}/Train/train.json", "train", True)
DEV_ALL   = load_split(f"{P4}/Train/dev.json",   "dev",   True)
TEST_ALL  = load_split(f"{P4}/Test/test.json",   "test",  False)

for name, ds in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
    nc = np.asarray([d["nchar"] for d in ds])
    msg = f"{name:5s}: {len(ds):6d} cases | chars/case mean {nc.mean():9.0f} median {np.median(nc):9.0f} p90 {np.percentile(nc, 90):9.0f} max {nc.max():9d}"
    if ds[0]["y"] is not None: msg += f" | label counts {dict(sorted(Counter(d['y'] for d in ds).items()))}"
    print(msg)
for name, ds in (("dev", DEV_ALL), ("test", TEST_ALL)):
    ids = [str(d["id"]) for d in ds]
    if len(set(ids)) != len(ids): print(f"WARNING: {name} ids are not unique -> paste this to Claude (the submission needs one prediction per id)")
_tr_ids, _te_ids = {str(d["id"]) for d in TRAIN_ALL}, {str(d["id"]) for d in TEST_ALL}
print("train/test id overlap:", len(_tr_ids & _te_ids))

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/train.json: list of 37387 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/dev.json: list of 1988 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Test/test.json: list of 1517 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train:  37387 cases | chars/case mean     19531 median     13023 p90     38996 max   2492701 | label counts {0: 22067, 1: 15320}
dev  :   1988 cases | chars/case mean     22110 median     16700 p90     40495 max    260198 | label counts {0: 994, 1: 994}
test :   1517 cases | chars/case mean     22698 median     17320 p90     42859 max    233000
train/test id overlap: 0


In [15]:
# Cell 14 — Length statistics: how much of a case does the chunk budget cover? (tokenises a small random sample of full texts)
_rng = random.Random(SEED)
_sample = _rng.sample(TRAIN_ALL, min(150, len(TRAIN_ALL)))
_t0 = time.time()
_tok_lens = np.asarray([len(tokenizer.encode(d["text"])) for d in _sample])
_dt = time.time() - _t0
_chars = np.asarray([d["nchar"] for d in _sample])
print(f"sample of {len(_sample)} train cases | chars/token {(_chars.sum() / _tok_lens.sum()):.2f} | full-text tokenisation {_chars.sum() / _dt / 1e6:.2f} M chars/s "
      f"({len(_sample) / _dt:.1f} cases/s serial)")
print(f"tokens/case: median {np.median(_tok_lens):.0f} | p75 {np.percentile(_tok_lens, 75):.0f} | p90 {np.percentile(_tok_lens, 90):.0f} | "
      f"p99 {np.percentile(_tok_lens, 99):.0f} | max {_tok_lens.max()}")
_budget = MAX_CHUNKS * BODY
print(f"budget {MAX_CHUNKS} x {BODY} = {_budget} tokens: the whole case fits for {100 * (_tok_lens <= _budget).mean():.0f}% of the sample; "
      f"on average the view reads {100 * np.mean(np.minimum(_tok_lens, _budget) / _tok_lens):.0f}% of a case's tokens")
_est_full = len(TRAIN_ALL) / (len(_sample) / _dt)
print(f"serial full-text tokenisation of all train cases would take ~{_est_full / 60:.0f} min; "
      f"DOC_VIEW={DOC_VIEW!r} " + ("tokenises only slices of the text (much faster)." if DOC_VIEW != "spread" else "needs the full text (use N_PROC > 1)."))
if _budget < np.median(_tok_lens): print("note: the budget is below the median case length; that is the point of the view ablation (what part to read).")
_ = _sample, _tok_lens, _chars

sample of 150 train cases | chars/token 4.66 | full-text tokenisation 3.73 M chars/s (181.5 cases/s serial)
tokens/case: median 2752 | p75 4880 | p90 6928 | p99 41351 | max 88617
budget 4 x 511 = 2044 tokens: the whole case fits for 38% of the sample; on average the view reads 70% of a case's tokens
serial full-text tokenisation of all train cases would take ~3 min; DOC_VIEW='tail' tokenises only slices of the text (much faster).
note: the budget is below the median case length; that is the point of the view ablation (what part to read).


## Selecting and tokenising the chunks
Only what the view needs is tokenised: for `tail` / `head` / `head_tail` the notebook tokenises a **character slice** of `CHAR_PER_TOK` characters per wanted token (doubling it if the slice holds too few tokens), which is orders of magnitude cheaper than tokenising a whole judgment. Chunks are aligned to the **end** of the text for the tail (so the last chunk always ends exactly at the end of the case) and to the start for the head. Results are cached on disk and tokenisation runs in `N_PROC` worker processes.

In [16]:
# Cell 15 — Chunk selection (views), tokenisation, cache
def _enc(s):
    return list(tokenizer.encode(s))

def _chunks(ids, from_end):
    """Split a token list into pieces of BODY tokens; alignment from the end (tail) or from the start (head)."""
    n, out = len(ids), []
    if from_end:
        e = n
        while e > 0:
            s = max(0, e - BODY); out.append(ids[s:e]); e = s
        out.reverse()
    else:
        for s in range(0, n, BODY): out.append(ids[s:s + BODY])
    return out

def _slice_tokens(text, need, side):
    """First / last `need` tokens of `text`, tokenising only a character slice. Returns (tokens, text_was_fully_tokenised)."""
    m = need * CHAR_PER_TOK
    while True:
        if m >= len(text):
            ids = _enc(text); return (ids[:need] if side == "head" else ids[-need:]), True
        if side == "head":
            s = text[:m]; j = s.rfind(" "); s = s[:j] if j > m // 2 else s            # drop a cut-off last word
        else:
            s = text[-m:]; j = s.find(" "); s = s[j + 1:] if 0 <= j < m // 2 else s   # drop a cut-off first word
        ids = _enc(s)
        if len(ids) >= need + 8:
            return (ids[:need] if side == "head" else ids[-need:]), False
        m *= 2

def _view_from_full(ids):
    K = MAX_CHUNKS
    if DOC_VIEW == "tail":  return _chunks(ids, True)[-K:]
    if DOC_VIEW == "head":  return _chunks(ids, False)[:K]
    if DOC_VIEW == "head_tail":
        if len(ids) <= K * BODY: return _chunks(ids, False)
        Kh = (K + 1) // 2; Kt = K - Kh
        return _chunks(ids[:Kh * BODY], False) + _chunks(ids[-Kt * BODY:], True)
    P = _chunks(ids, False)                                                          # "spread": K chunks evenly over the whole case
    if len(P) <= K: return P
    return [P[i] for i in sorted(set(np.linspace(0, len(P) - 1, K).round().astype(int).tolist()))]

def select_chunks(text):
    K, n = MAX_CHUNKS, len(text)
    if DOC_VIEW == "spread" or n <= K * BODY * CHAR_PER_TOK:
        pieces = _view_from_full(_enc(text))
    elif DOC_VIEW == "tail":
        ids, _ = _slice_tokens(text, K * BODY, "tail"); pieces = _chunks(ids, True)[-K:]
    elif DOC_VIEW == "head":
        ids, _ = _slice_tokens(text, K * BODY, "head"); pieces = _chunks(ids, False)[:K]
    else:                                                                            # head_tail
        Kh = (K + 1) // 2; Kt = K - Kh
        h, c1 = _slice_tokens(text, Kh * BODY, "head"); t, c2 = _slice_tokens(text, Kt * BODY, "tail")
        pieces = _view_from_full(_enc(text)) if (c1 or c2) else _chunks(h, False)[:Kh] + _chunks(t, True)[-Kt:]
    if not pieces: pieces = [[EOS_ID]]                                               # an empty case still needs one token
    return pieces

_TXT = None
def _work(i):
    return [np.asarray([BOS_ID] + list(p), dtype=np.int32) for p in select_chunks(_TXT[i])]

def tokenize_docs(docs, cache_name):
    """Adds d['chunks'] = list of int32 arrays (each starts with [BOS], length <= CHUNK_TOKENS). Cached on disk."""
    global _TXT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"{cache_name}_v{VOCAB_SIZE}_{DOC_VIEW}_c{CHUNK_TOKENS}_k{MAX_CHUNKS}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None, None))[0] != d["nchar"]]
    if todo:
        _TXT = [d["text"] for d in docs]
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, todo, chunksize=8), total=len(todo), desc=f"tokenise {cache_name}", leave=False))
            except Exception as ex:
                print("parallel tokenisation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(todo, desc=f"tokenise {cache_name}", leave=False)]
        for i, r in zip(todo, res): cache[docs[i]["uid"]] = (docs[i]["nchar"], r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d["chunks"] = cache[d["uid"]][1]
    return docs

def chunk_stats(docs, title):
    nch = np.asarray([len(d["chunks"]) for d in docs]); ntok = np.asarray([sum(len(c) for c in d["chunks"]) for d in docs])
    print(f"{title}: {len(docs)} cases | chunks/case mean {nch.mean():.2f} (full {MAX_CHUNKS}: {100 * (nch == MAX_CHUNKS).mean():.0f}%) | "
          f"tokens/case read mean {ntok.mean():.0f}")

def slot_offset(n_chunks):
    """Slot of the first chunk. For the tail view the chunks are RIGHT-aligned, so the last chunk (end of the judgment) is always slot K-1."""
    return (MAX_CHUNKS - n_chunks) if DOC_VIEW == "tail" else 0

def build_batch(docs, idxs, with_labels=True):
    """-> ids [N, L], mask [N, L], owner [N] (case position in the batch), slot [N], y [B]."""
    seqs, owner, slot, ys = [], [], [], []
    for b, di in enumerate(idxs):
        d = docs[di]; off = slot_offset(len(d["chunks"]))
        for j, c in enumerate(d["chunks"]):
            seqs.append(c); owner.append(b); slot.append(off + j)
        if with_labels: ys.append(d["y"])
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), L), dtype=np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
    y = np.asarray(ys, dtype=np.int64) if with_labels else None
    return ids, mask, np.asarray(owner), np.asarray(slot), y

## Metrics, decision threshold (pure numpy)
Binary macro-F1 = mean of the F1 of class 0 and class 1. The classifier outputs `log p(1) − log p(0)` (the logit margin); predicting `1` when `margin + b > 0` with a tuned `b` is the binary version of the per-class bias tuning of Phase 3. `b` is kept only if a 2-fold, label-stratified check on dev (tune on one half, score the other) shows a real gain.

In [17]:
# Cell 16 — Metrics (macro-F1), confusion, threshold tuning
def bin_scores(y, p):
    y = np.asarray(y, dtype=np.int64); p = np.asarray(p, dtype=np.int64)
    cm = np.zeros((2, 2), dtype=np.int64); np.add.at(cm, (y, p), 1)                  # cm[gold, pred]
    tp = np.diag(cm).astype(np.float64); sup = cm.sum(1).astype(np.float64); npred = cm.sum(0).astype(np.float64)
    P = np.divide(tp, npred, out=np.zeros(2), where=npred > 0)
    R = np.divide(tp, sup,   out=np.zeros(2), where=sup > 0)
    Fs = np.divide(2 * P * R, P + R, out=np.zeros(2), where=(P + R) > 0)
    return dict(P=P, R=R, F=Fs, support=sup, cm=cm, accuracy=float(tp.sum() / max(cm.sum(), 1)),
                macro_f1=float(Fs.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()))

def margin(logp):
    return logp[:, 1] - logp[:, 0]

def evaluate(docs, logp, bias=0.0):
    y = np.asarray([d["y"] for d in docs])
    return bin_scores(y, (margin(logp) + bias > 0).astype(np.int64))

def tune_threshold(docs, logp, grid=np.arange(-3.0, 3.01, 0.1)):
    """Best additive bias b on the margin (ties -> the smallest |b|)."""
    y = np.asarray([d["y"] for d in docs]); m = margin(logp)
    base = bin_scores(y, (m > 0).astype(np.int64))["macro_f1"]; best_b, best = 0.0, base
    for b in sorted(grid.tolist(), key=abs):
        s = bin_scores(y, (m + b > 0).astype(np.int64))["macro_f1"]
        if s > best + 1e-9: best_b, best = float(b), s
    return best_b, base, best

def cv_threshold_gain(docs, logp, folds=2):
    """Honest check: tune on half of the dev cases, score the other half (label-stratified split); mean held-out gain."""
    y = np.asarray([d["y"] for d in docs]); fold = np.zeros(len(docs), dtype=int)
    for c in (0, 1):
        idx = np.where(y == c)[0]; fold[idx] = np.arange(len(idx)) % folds
    gains = []
    for f in range(folds):
        tr, te = np.where(fold != f)[0], np.where(fold == f)[0]
        b, _, _ = tune_threshold([docs[i] for i in tr], logp[tr])
        base = evaluate([docs[i] for i in te], logp[te])["macro_f1"]; tuned = evaluate([docs[i] for i in te], logp[te], b)["macro_f1"]
        gains.append(tuned - base)
    return float(np.mean(gains))

## Model, adaptation methods (LoRMA / LoRA / full / norm-only), training
The bidirectional-attention patch and the LoRA / LoRMA layers are the Phase-3 code, unchanged. New: the **document model** (chunk encoder → chunk pooling → aggregator → head), which replaces the sentence-pooling head.

In [18]:
# Cell 17 — Attention modes, the document model, LoRA / rsLoRA, and the adaptation methods
import copy

# ---- (a) Attention modes (as in the Phase-3 Dec-03 notebook; here with the key-padding mask only) ---------------------------------------------------------------------------------------------------
# Phase-1 Backbone builds ONE additive bias [B,H,T,S] = ALiBi (-m_h (i-j)) + causal (-inf for j>i) + padding, and gives it to
# every block. Here we replace Backbone.forward by a version that can hand DIFFERENT biases to different blocks:
#   blocks < bidir_from : the original causal ALiBi bias           (frozen lower blocks keep their Phase-1 behaviour)
#   blocks >= bidir_from: symmetric ALiBi  -m_h |i-j|, no causal part (right context is visible)
def _bidir_alibi_bias(bb, B, T, attention_mask, dev):
    pos = torch.arange(T, device=dev)
    dist = (pos[:, None] - pos[None, :]).abs().float()                           # |i - j|
    bias = (-bb.alibi.slopes.to(dev)[:, None, None] * dist).unsqueeze(0)         # [1, H, T, T]
    if attention_mask is not None:                                               # mask padded KEYS (real keys always remain, so no NaN rows)
        bias = bias.expand(B, -1, -1, -1).masked_fill(~attention_mask.bool()[:, None, None, :], float("-inf"))
    return bias

def _hybrid_forward(self, input_ids, attention_mask=None, kv_caches=None):
    assert kv_caches is None, "the CJPE notebook never uses a KV cache"
    B, T = input_ids.shape
    dev = input_ids.device
    x = self.emb_drop(self.tok_emb(input_ids))
    bias_causal = Backbone._attn_bias(self, B, T, 0, attention_mask, dev) if self.bidir_from > 0 else None
    bias_bidir  = _bidir_alibi_bias(self, B, T, attention_mask, dev) if self.bidir_from < len(self.blocks) else None
    for i, blk in enumerate(self.blocks):
        x, _ = blk(x, attn_bias=bias_bidir if i >= self.bidir_from else bias_causal, rope_cs=None, kv_cache=None)
    return self.norm_f(x), []

def set_attention_mode(bb):
    """Install the attention mode on a Backbone instance (sets bb.bidir_from; the forward is the hybrid one for every mode)."""
    n = len(bb.blocks)
    if ATTN_MODE == "bidir":
        bb.bidir_from = 0
    elif ATTN_MODE == "bidir_top":
        bb.bidir_from = max(0, n - BIDIR_TOP_K)
    else:
        bb.bidir_from = n                                          # "causal": every block keeps the Phase-1 causal ALiBi bias
    bb.forward = types.MethodType(_hybrid_forward, bb)
    print("attention: all blocks causal (Phase-1)" if bb.bidir_from >= n else
          "attention: all blocks bidirectional (symmetric ALiBi)" if bb.bidir_from == 0 else
          f"attention: blocks 0..{bb.bidir_from - 1} causal | blocks {bb.bidir_from}..{n - 1} bidirectional (symmetric ALiBi)")



# ---- (b) Document model: Phase-1 backbone per chunk -> mean-pool -> aggregator over chunks -> Linear(d, 2) ------------------
class DocModel(nn.Module):
    """Every chunk goes through the (adapted) Phase-1 backbone independently; its token vectors are mean-pooled (padding excluded).
    The chunk vectors of a case sit in slots 0..K-1 (right-aligned for the tail view) and are aggregated by AGG."""
    def __init__(self, backbone, d_model, n_slots, agg, dropout, hier_layers):
        super().__init__()
        self.backbone, self.agg, self.K = backbone, agg, n_slots
        if agg == "attn":
            self.att_w = nn.Linear(d_model, d_model); self.att_v = nn.Linear(d_model, 1, bias=False)
        elif agg == "hier":
            self.slot_emb = nn.Embedding(n_slots, d_model); nn.init.normal_(self.slot_emb.weight, 0.0, 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model)); self.ln_in = nn.LayerNorm(d_model); self.ln_out = nn.LayerNorm(d_model)
            layer = nn.TransformerEncoderLayer(d_model, 8, dim_feedforward=1024, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, hier_layers, enable_nested_tensor=False)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 2)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def encode(self, ids, mask):
        """Same computation as the hybrid Backbone.forward above (per-block ALiBi bias, no KV cache), with optional activation checkpointing."""
        bb = self.backbone; N, T = ids.shape; dev = ids.device
        x = bb.emb_drop(bb.tok_emb(ids))
        b_c = Backbone._attn_bias(bb, N, T, 0, mask, dev) if bb.bidir_from > 0 else None
        b_b = _bidir_alibi_bias(bb, N, T, mask, dev) if bb.bidir_from < len(bb.blocks) else None
        for i, blk in enumerate(bb.blocks):
            bias = b_b if i >= bb.bidir_from else b_c
            if self.training and GRAD_CKPT:
                x = checkpoint(lambda t, blk=blk, bias=bias: blk(t, attn_bias=bias, rope_cs=None, kv_cache=None)[0], x, use_reentrant=False)
            else:
                x, _ = blk(x, attn_bias=bias, rope_cs=None, kv_cache=None)
        return bb.norm_f(x)

    def forward(self, ids, mask, owner, slot, B):
        h = self.encode(ids, mask).float()                                       # [N, T, d]
        m = mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp_min(1.0)                        # [N, d]   chunk vectors
        D = pooled.size(-1)
        grid = pooled.new_zeros(B, self.K, D); valid = torch.zeros(B, self.K, dtype=torch.bool, device=ids.device)
        grid[owner, slot] = pooled; valid[owner, slot] = True
        alpha = None
        if self.agg == "mean":
            v = valid.unsqueeze(-1).float(); doc = (grid * v).sum(1) / v.sum(1).clamp_min(1.0)
        elif self.agg == "attn":
            s = self.att_v(torch.tanh(self.att_w(grid))).squeeze(-1).masked_fill(~valid, float("-inf"))
            alpha = s.softmax(-1); doc = (alpha.unsqueeze(-1) * grid).sum(1)
        else:
            z = self.ln_in(grid) + self.slot_emb(torch.arange(self.K, device=ids.device)).unsqueeze(0)
            seq = torch.cat([self.cls.expand(B, 1, D), z], dim=1)
            kpm = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=ids.device), ~valid], dim=1)
            doc = self.ln_out(self.enc(seq, src_key_padding_mask=kpm)[:, 0])
        return self.head(self.drop(doc)), alpha


# ---- (c) LoRA and rank-stabilised LoRA ---------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + gamma * B(A(dropout(x))).  W frozen; A ~ Kaiming-uniform, B = 0, so the update starts at exactly 0.
       standard LoRA:  gamma = alpha / r
       rsLoRA:         gamma = alpha / sqrt(r)   (Kalajdzievski 2023: the unique scaling for which learning is stable for every rank)"""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float, rank_stabilized: bool = False):
        super().__init__()
        self.base, self.r = base, r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n



def scaling_demo(ranks=(4, 16, 64, 256), d=512, alpha=16.0, eta=1.0, trials=400, seed=0):
    """One SGD step on B from B = 0 with a linear loss L = g.h:  Delta h = -eta * gamma^2 * ||A x||^2 * g, with ||A x||^2 ~ r.
    Prints |Delta h| (relative to rank 4) for gamma = alpha/r (LoRA) and gamma = alpha/sqrt(r) (rsLoRA)."""
    gen = torch.Generator().manual_seed(seed)
    bound = 1.0 / math.sqrt(d)                                   # Kaiming-uniform(a=sqrt(5)) bound for fan_in = d
    rows = []
    for r in ranks:
        lo, rs = [], []
        for _ in range(trials):
            x = torch.randn(d, generator=gen); g = torch.randn(d, generator=gen); g = g / g.norm()
            A = (torch.rand(r, d, generator=gen) * 2 - 1) * bound
            Ax2 = float((A @ x).pow(2).sum())
            lo.append(eta * (alpha / r) ** 2 * Ax2); rs.append(eta * (alpha / math.sqrt(r)) ** 2 * Ax2)   # |Delta h| (|g| = 1)
        rows.append((r, float(np.mean(lo)), float(np.mean(rs))))
    b_lo, b_rs = rows[0][1], rows[0][2]
    print("one-step output change |dh| relative to rank 4  (theory: LoRA ~ 1/r, rsLoRA ~ constant)")
    print(f"  {'rank r':>7s} | {'LoRA  gamma=a/r':>16s} {'|dh| rel.':>10s} | {'rsLoRA gamma=a/sqrt(r)':>23s} {'|dh| rel.':>10s}")
    for r, lo, rs in rows:
        print(f"  {r:7d} | {alpha / r:16.4f} {lo / b_lo:10.3f} | {alpha / math.sqrt(r):23.4f} {rs / b_rs:10.3f}")

if SCALING_DEMO:
    scaling_demo()


def build_model():
    """Fresh copy of the Phase-1 weights every call (several runs in one session do not contaminate each other)."""
    backbone = copy.deepcopy(lm.backbone).cpu()
    ref = copy.deepcopy(backbone) if METHOD in ("lora", "rslora") else None    # untouched copy for the identity check
    set_attention_mode(backbone)
    if ref is not None: set_attention_mode(ref)
    model = DocModel(backbone, lm_cfg.d_model, MAX_CHUNKS, AGG, HEAD_DROPOUT, HIER_LAYERS)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # everything trainable
    elif METHOD == "partial":
        n = len(backbone.blocks)
        for p in backbone.parameters(): p.requires_grad = False                 # embeddings + all blocks frozen ...
        for blk in backbone.blocks[n - PARTIAL_K:]:                             # ... except the top K blocks
            for p in blk.parameters(): p.requires_grad = True
        for p in backbone.norm_f.parameters(): p.requires_grad = True           # final RMSNorm feeds the aggregator/head
        print(f"partial fine-tuning: blocks {n - PARTIAL_K}..{n - 1} ({PARTIAL_K} of {n}) + final norm + aggregator/head trainable; "
              f"embeddings and blocks 0..{n - PARTIAL_K - 1} frozen")
        if ATTN_MODE == "bidir":
            print("WARNING: ATTN_MODE='bidir' unmasks the FROZEN lower blocks too (they see an attention pattern they were never "
                  "trained for and cannot adapt). 'bidir_top' keeps them exactly as in Phase 1.")
    elif METHOD in ("lora", "rslora"):
        for p in backbone.parameters(): p.requires_grad = False
        rs = METHOD == "rslora"
        n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"], rank_stabilized=rs))
        gamma = hp["alpha"] / math.sqrt(hp["r"]) if rs else hp["alpha"] / hp["r"]
        print(f"{'rsLoRA' if rs else 'LoRA'} applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']}, gamma={gamma:.4f})")
        # The adapted model must equal the Phase-1 model at initialisation (B = 0), in the attention mode that is actually used.
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a, _ = backbone.eval()(_ids, attention_mask=_m); b, _ = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        backbone.train(); del ref
    elif METHOD == "norm_only":
        for p in backbone.parameters(): p.requires_grad = False
        n = 0
        for m in backbone.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_new = sum(p.numel() for n_, p in model.named_parameters() if not n_.startswith("backbone."))
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | of which new (aggregator + head) {n_new/1e6:.3f}M")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.bincount([d["y"] for d in train_docs], minlength=2).astype(np.float64)
    w = np.ones(2)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

one-step output change |dh| relative to rank 4  (theory: LoRA ~ 1/r, rsLoRA ~ constant)
   rank r |  LoRA  gamma=a/r  |dh| rel. |  rsLoRA gamma=a/sqrt(r)  |dh| rel.
        4 |           4.0000      1.000 |                  8.0000      1.000
       16 |           1.0000      0.253 |                  4.0000      1.011
       64 |           0.2500      0.061 |                  2.0000      0.974
      256 |           0.0625      0.015 |                  1.0000      0.985


In [19]:
# Cell 18 — Inference: cases -> log-probabilities (+ the attention profile over chunk slots)
@torch.no_grad()
def predict_logp(model, docs, batch_docs=None):
    """-> (logp [n, 2], alpha [n, K] or None, seconds). alpha[i, s] = attention weight of slot s (0 where the slot is empty)."""
    model.eval(); bs = batch_docs or 2 * BATCH_DOCS
    order = sorted(range(len(docs)), key=lambda i: sum(len(c) for c in docs[i]["chunks"]))     # similar lengths together
    out = np.zeros((len(docs), 2), dtype=np.float32); al = np.zeros((len(docs), MAX_CHUNKS), dtype=np.float32)
    t0 = time.time()
    for i in range(0, len(order), bs):
        idxs = order[i:i + bs]
        ids, mask, owner, slot, _ = build_batch(docs, idxs, with_labels=False)
        ids, mask = torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)
        owner, slot = torch.from_numpy(owner).to(device), torch.from_numpy(slot).to(device)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, alpha = model(ids, mask, owner, slot, len(idxs))
        out[idxs] = logits.float().log_softmax(-1).cpu().numpy()
        if alpha is not None: al[idxs] = alpha.float().cpu().numpy()
    if device.type == "cuda": torch.cuda.synchronize()
    return out, (al if model.agg == "attn" else None), time.time() - t0

In [20]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def train_doc_model(model, train_docs, val_docs, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, counts = class_weights(train_docs)
    print("class counts:", counts.astype(int).tolist(), "| class weights:", np.round(cw.cpu().numpy(), 3).tolist())
    new_p = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("backbone.") and p.requires_grad:
            (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=new_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(train_docs)
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH_DOCS) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} cases/epoch | {steps_per_epoch} optimiser steps/epoch (effective batch {BATCH_DOCS * ACCUM}) | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time(); n_seen = 0
    for ep in range(1, epochs + 1):
        order = list(range(n_items)); random.Random(SEED + ep).shuffle(order)
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(n_items / BATCH_DOCS)
        for bi in range(nb):
            idxs = order[bi * BATCH_DOCS:(bi + 1) * BATCH_DOCS]
            ids, mask, owner, slot, y = build_batch(train_docs, idxs)
            ids, mask, owner, slot, y = (torch.from_numpy(x).to(device) for x in (ids, mask, owner, slot, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits, _ = model(ids, mask, owner, slot, len(idxs))
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1; n_seen += len(idxs)
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step == 10:
                    per = (time.time() - t0) / 10
                    print(f"  >> speed: {per:.1f} s/step | {BATCH_DOCS * ACCUM / per:.1f} cases/s | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus dev evaluation)")
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25 * ACCUM:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping (the best epoch so far is kept)"); done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
            rec.update(val_macro_f1=ev["macro_f1"], val_acc=ev["accuracy"], val_P1=float(ev["P"][1]), val_R1=float(ev["R"][1]),
                       val_frac_pred1=float((lp[:, 1] > lp[:, 0]).mean()))
            if ev["macro_f1"] > best["score"]:
                best = dict(score=ev["macro_f1"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, cases_seen=n_seen,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [21]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TRAIN_ALL, min(SMOKE_TRAIN_DOCS, len(TRAIN_ALL)))
    val_docs = _r.sample(DEV_ALL, min(SMOKE_VAL_DOCS, len(DEV_ALL)))
elif FINAL_FIT:
    train_docs = TRAIN_ALL + DEV_ALL; val_docs = []
else:
    train_docs = TRAIN_ALL; val_docs = DEV_ALL
_t0 = time.time()
tokenize_docs(train_docs, "train_smoke" if SMOKE_TEST else ("train_dev" if FINAL_FIT else "train"))
if val_docs: tokenize_docs(val_docs, "dev_smoke" if SMOKE_TEST else "dev")
print(f"tokenisation {time.time() - _t0:.0f}s")
chunk_stats(train_docs, "train")
if val_docs: chunk_stats(val_docs, "dev")

if val_docs:                                        # reference points: constant predictions
    for c in (0, 1):
        _ev = evaluate(val_docs, np.log(np.tile(np.eye(2, dtype=np.float32)[c] * 0.998 + 0.001, (len(val_docs), 1))))
        print(f"constant-'{c}' baseline: dev macro-F1 {_ev['macro_f1']:.4f} | accuracy {_ev['accuracy']:.3f}")

model = build_model()
history, best, stats = train_doc_model(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev macro-F1 {best['score']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

tokenise train:   0%|          | 0/37387 [00:00<?, ?it/s]

tokenise dev:   0%|          | 0/1988 [00:00<?, ?it/s]

tokenisation 49s
train: 37387 cases | chunks/case mean 3.42 (full 4: 71%) | tokens/case read mean 1659
dev: 1988 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1949
constant-'0' baseline: dev macro-F1 0.3333 | accuracy 0.500
constant-'1' baseline: dev macro-F1 0.3333 | accuracy 0.500
attention: all blocks bidirectional (symmetric ALiBi)
attention: all blocks bidirectional (symmetric ALiBi)
rsLoRA applied to 104 Linear layers (r=64, alpha=16, gamma=2.0000)
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 62.24M | trainable 8.358M (13.43%) | of which new (aggregator + head) 0.264M
class counts: [22067, 15320] | class weights: [1.0, 1.2000000476837158]
37387 cases/epoch | 2337 optimiser steps/epoch (effective batch 16) | total 9348 steps
  >> speed: 1.4 s/step | 11.9 cases/s | projected 52.6 min/epoch (210 min for 4 epochs, plus dev evaluation)
  ep 1 step 25/9348 | loss 0.7466 | lr x0.045 | 33s
  ep 1 step 50/9348 | loss 0.6946 | lr x0.089 | 67

In [22]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
    frac1 = float((lp[:, 1] > lp[:, 0]).mean())
    print(f"loss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | dev macro-F1 {ev['macro_f1']:.4f} | share predicted '1': {frac1:.2f}")
    ok = drop < 0.95 and 0.02 < frac1 < 0.98
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or one class dominates; ~150 steps is short for an adapter - compare METHOD='full' before debugging")
    print("Check the '>> speed' line above: it projects the minutes per epoch of the full run; set EPOCHS / MAX_CHUNKS / TIME_BUDGET_MIN accordingly.")
else:
    print("not a smoke run")

not a smoke run


In [23]:
# Cell 22 — Dev analysis: scores, per-class table, confusion, threshold tuning, FP/FN by length, attention profile (full runs with a dev set)
THRESH = 0.0 if THRESH_FINAL is None else float(THRESH_FINAL)
dev_ev = dev_ev_tuned = None; thresh_gain = 0.0; dev_extra = {}
if not SMOKE_TEST and val_docs:
    logp_val, alpha_val, t_inf = predict_logp(model, val_docs)
    dev_ev = evaluate(val_docs, logp_val)
    print(f"===== dev | macro-F1 {dev_ev['macro_f1']:.4f} | macro-P {dev_ev['macro_p']:.4f} | macro-R {dev_ev['macro_r']:.4f} | accuracy {dev_ev['accuracy']:.4f}   <- report the UNTUNED macro-F1")
    for c in (0, 1):
        print(f"  class {c}: support {int(dev_ev['support'][c]):5d} | P {dev_ev['P'][c]:.3f} | R {dev_ev['R'][c]:.3f} | F1 {dev_ev['F'][c]:.3f}")
    print("  confusion [gold x pred]:", dev_ev["cm"].tolist())
    print(f"  inference: {t_inf:.1f}s for {len(val_docs)} cases = {1000 * t_inf / len(val_docs):.0f} ms/case ({MAX_CHUNKS} chunks x {CHUNK_TOKENS} tokens max)")
    dev_extra["infer_ms_per_case"] = 1000 * t_inf / len(val_docs)

    if THRESH_TUNE:
        b_t, s0, s1 = tune_threshold(val_docs, logp_val)
        cv_gain = cv_threshold_gain(val_docs, logp_val) if len(val_docs) >= 20 else 0.0
        thresh_gain = cv_gain
        print(f"threshold tuning on dev: {s0:.4f} -> {s1:.4f} (in-sample gain {s1 - s0:+.4f}, optimistic) | 2-fold held-out gain {cv_gain:+.4f} | bias = {b_t:+.2f}")
        if cv_gain >= THRESH_MIN_GAIN:
            THRESH = b_t; dev_ev_tuned = evaluate(val_docs, logp_val, THRESH)
            print(f"  kept. tuned dev macro-F1 {dev_ev_tuned['macro_f1']:.4f}. For FINAL_FIT=True set  THRESH_FINAL = {THRESH:+.2f}")
        else:
            print(f"  held-out gain below THRESH_MIN_GAIN={THRESH_MIN_GAIN}: threshold NOT used")

    # --- characteristics of the errors: FP / FN by case length (character terciles) ---
    y_v = np.asarray([d["y"] for d in val_docs]); p_v = (margin(logp_val) + THRESH > 0).astype(int)
    nchar_v = np.asarray([d["nchar"] for d in val_docs]); cuts = np.percentile(nchar_v, [33.3, 66.7])
    bucket = np.digitize(nchar_v, cuts); names = ["short", "medium", "long"]
    rows = []
    print("\nerrors by case length (terciles of characters):")
    print(f"  {'bucket':8s} {'cases':>6s} {'median chars':>13s} {'acc':>6s} {'macroF1':>8s} {'FP':>5s} {'FN':>5s} {'frac gold=1':>12s}")
    for bk in range(3):
        sel = bucket == bk
        if sel.sum() == 0: continue
        s = bin_scores(y_v[sel], p_v[sel]); fp = int(((p_v == 1) & (y_v == 0) & sel).sum()); fn = int(((p_v == 0) & (y_v == 1) & sel).sum())
        rows.append(dict(bucket=names[bk], n=int(sel.sum()), median_chars=float(np.median(nchar_v[sel])), acc=s["accuracy"], macro_f1=s["macro_f1"], fp=fp, fn=fn, frac_gold1=float(y_v[sel].mean())))
        print(f"  {names[bk]:8s} {int(sel.sum()):6d} {np.median(nchar_v[sel]):13.0f} {s['accuracy']:6.3f} {s['macro_f1']:8.3f} {fp:5d} {fn:5d} {y_v[sel].mean():12.3f}")
    dev_extra["by_length"] = rows
    m_v = margin(logp_val) + THRESH
    dev_extra["mean_margin"] = dict(TP=float(m_v[(y_v == 1) & (p_v == 1)].mean()) if ((y_v == 1) & (p_v == 1)).any() else None,
                                    FP=float(m_v[(y_v == 0) & (p_v == 1)].mean()) if ((y_v == 0) & (p_v == 1)).any() else None,
                                    FN=float(m_v[(y_v == 1) & (p_v == 0)].mean()) if ((y_v == 1) & (p_v == 0)).any() else None,
                                    TN=float(m_v[(y_v == 0) & (p_v == 0)].mean()) if ((y_v == 0) & (p_v == 0)).any() else None)
    print("mean logit margin by outcome (a confident error has a large |margin|):", {k: (None if v is None else round(v, 2)) for k, v in dev_extra["mean_margin"].items()})
    errs = [dict(id=str(val_docs[i]["id"]), gold=int(y_v[i]), pred=int(p_v[i]), margin=float(m_v[i]), nchar=int(nchar_v[i]), n_chunks=len(val_docs[i]["chunks"]))
            for i in np.where(p_v != y_v)[0]]
    (RUN_DIR / "dev_errors.json").write_text(json.dumps(errs))

    # --- which chunk does the model rely on? mean attention weight per slot (AGG='attn') ---
    if alpha_val is not None:
        prof = alpha_val.mean(0); cnt = (alpha_val > 0).mean(0)
        where = {"tail": "slot K-1 = end of the judgment", "head": "slot 0 = start", "head_tail": "first slots = start, last slots = end", "spread": "slot 0 = start ... K-1 = end"}[DOC_VIEW]
        print(f"\nmean attention weight per chunk slot ({where}):", np.round(prof, 3).tolist(), "| share of cases with that slot filled:", np.round(cnt, 2).tolist())
        full = np.asarray([len(d["chunks"]) == MAX_CHUNKS for d in val_docs])
        if full.any(): print("   ... on cases with all slots filled:", np.round(alpha_val[full].mean(0), 3).tolist())
        dev_extra["attn_profile"] = prof.tolist(); dev_extra["attn_profile_full_cases"] = alpha_val[full].mean(0).tolist() if full.any() else None
else:
    print("no dev analysis (smoke run or FINAL_FIT)" + ("" if THRESH_FINAL is None else f" | using THRESH_FINAL={THRESH:+.2f}"))

===== dev | macro-F1 0.3635 | macro-P 0.5344 | macro-R 0.5040 | accuracy 0.5040   <- report the UNTUNED macro-F1
  class 0: support   994 | P 0.502 | R 0.974 | F1 0.663
  class 1: support   994 | P 0.567 | R 0.034 | F1 0.065
  confusion [gold x pred]: [[968, 26], [960, 34]]
  inference: 73.0s for 1988 cases = 37 ms/case (4 chunks x 512 tokens max)
threshold tuning on dev: 0.3635 -> 0.5084 (in-sample gain +0.1449, optimistic) | 2-fold held-out gain +0.1449 | bias = +0.20
  kept. tuned dev macro-F1 0.5084. For FINAL_FIT=True set  THRESH_FINAL = +0.20

errors by case length (terciles of characters):
  bucket    cases  median chars    acc  macroF1    FP    FN  frac gold=1
  short       662          8945  0.486    0.481   114   226        0.532
  medium      664         16700  0.482    0.470   226   118        0.494
  long        662         33629  0.559    0.557   184   108        0.474
mean logit margin by outcome (a confident error has a large |margin|): {'TP': 0.08, 'FP': 0.08, 'FN': -0

In [24]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, logp, bias, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pred = (margin(logp) + bias > 0).astype(int)
    sub = [dict(id=d["id"], pred=int(p)) for d, p in zip(docs, pred)]            # the id keeps the type it has in test.json
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} cases | predicted share of '1': {pred.mean():.3f}")
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, one integer 0/1 per official test id, no extras."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, pred}"
    want = [d["id"] for d in docs]
    got = [e["id"] for e in sub]
    assert len(got) == len(set(map(str, got))), "duplicate ids in the submission"
    assert set(map(str, got)) == set(map(str, want)), f"ids differ from test.json: {len(set(map(str, want)) - set(map(str, got)))} missing, {len(set(map(str, got)) - set(map(str, want)))} unknown"
    assert all(isinstance(e["pred"], int) and not isinstance(e["pred"], bool) and e["pred"] in (0, 1) for e in sub), "pred must be the integer 0 or 1"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} cases | first entries {sub[:2]}")
    print("   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).")

if not SMOKE_TEST:
    tokenize_docs(TEST_ALL, "test")
    chunk_stats(TEST_ALL, "test")
    logp_test, _, _ = predict_logp(model, TEST_ALL)
    submission = write_submission(TEST_ALL, logp_test, THRESH, RUN_DIR / "submission")
    _tr = np.bincount([d["y"] for d in train_docs], minlength=2); print(f"train share of '1': {_tr[1] / _tr.sum():.3f}")
else:
    print("smoke run: no submission written")

tokenise test:   0%|          | 0/1517 [00:00<?, ?it/s]

test: 1517 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1947
wrote /kaggle/working/runs/p4_dec03_rslora/submission/submission.zip | 1517 cases | predicted share of '1': 0.558
VERIFIED /kaggle/working/runs/p4_dec03_rslora/submission/submission.zip | 1517 cases | first entries [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}]
   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).
train share of '1': 0.410


In [25]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / representation tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, Path): return str(o)
    return o

def _ev_summary(ev):
    if ev is None: return None
    return dict(macro_f1=ev["macro_f1"], macro_p=ev["macro_p"], macro_r=ev["macro_r"], accuracy=ev["accuracy"],
                per_class={str(c): dict(support=int(ev["support"][c]), P=round(float(ev["P"][c]), 4), R=round(float(ev["R"][c]), 4), F1=round(float(ev["F"][c]), 4)) for c in (0, 1)},
                confusion=ev["cm"].tolist())

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
           hier_layers=HIER_LAYERS, final_fit=FINAL_FIT, smoke=SMOKE_TEST, mixer=lm_cfg.mixer_type, partial_k=PARTIAL_K, bidir_top_k=BIDIR_TOP_K, pos_type=lm_cfg.pos_type,
           hp=HP[METHOD], epochs=EPOCHS, batch_docs=BATCH_DOCS, accum=ACCUM, class_weight=CLASS_WEIGHT,
           params_total=model.n_total, params_trainable=model.n_trainable, history=history, best_epoch=best["epoch"], stats=stats,
           threshold=THRESH, threshold_gain=thresh_gain, dev_untuned=_ev_summary(dev_ev), dev_tuned=_ev_summary(dev_ev_tuned), dev_extra=dev_extra,
           n_train=len(train_docs), n_dev=len(val_docs))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "agg": AGG, "doc_view": DOC_VIEW, "log": _jsonable(log)},
           RUN_DIR / f"cjpe_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['cjpe_p4_dec03_rslora.pt', 'dev_errors.json', 'run_log.json', 'submission']


In [26]:
# Cell 25 — Export this run's document-level distributions ("soft labels") so it can serve as the KD teacher (run #5)
# All four Phase-1 models share one tokenizer, so these arrays can teach ANY student without this notebook's model code.
# Stored as float16 log-probabilities: softmax(logp / T) = softmax(z / T) because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time(); _out, _split, _ids = {}, {}, {}
    for _name, _docs in (("train", train_docs), ("dev", val_docs)):
        if not _docs: continue
        _lp, _, _ = predict_logp(model, _docs)
        for d, l in zip(_docs, _lp):
            _out[d["uid"]] = l.astype(np.float16); _split[d["uid"]] = _name; _ids[d["uid"]] = str(d["id"])
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS), best_epoch=int(best["epoch"]),
                 final_fit=FINAL_FIT, dev_macro_f1=None if dev_ev is None else float(dev_ev["macro_f1"]), threshold=float(THRESH))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, docs=_out, split=_split, ids=_ids), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.1f} MB) in {time.time()-_t0:.0f}s | {len(_out)} cases | teacher params {_meta['params_total']/1e6:.1f}M | dev macro-F1 {_meta['dev_macro_f1']}")
    print("caveat: on the TRAIN cases the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)


## What to do next

1. **Smoke test first:** `SMOKE_TEST=True` (a few minutes). Read the scaling table (LoRA's one-step update shrinks like 1/r, rsLoRA's stays flat), the *attention:* line (blocks 0–12 bidirectional), the *identity-at-init* line, the trainable-parameter count (about 8.1M for rsLoRA r = 64), the **length statistics** (Cell 14), the **`>> speed` line** (13 layers + the r = 64 adapters make this the slowest of the four backbones per token), and the smoke verdict.
2. **Use the same budget as runs #1–#3** (`DOC_VIEW`, `CHUNK_TOKENS`, `MAX_CHUNKS`, `AGG`, `EPOCHS`), so that the only differences are the backbone (ALiBi + MLA) and the technique (rsLoRA). If the epoch is too slow, set `TIME_BUDGET_MIN` or use `GRAD_CKPT=True` only if memory forces it (it costs ~30 % time).
3. **Full run:** `SMOKE_TEST=False`. It trains on train, selects the epoch on dev macro-F1, tunes the decision threshold (kept only if it helps on held-out dev halves) and **already writes a valid `submission.zip`**. Upload it to Codabench right away (evaluation takes 15-30 min).
4. **Controls for the report** (same notebook, change Cell 2 only, new `RUN_NAME` each time): **rsLoRA vs LoRA on the same backbone:** `METHOD="lora"` (r = 16, α = 32) and, to see the high-rank problem itself, `METHOD="lora"` with `HP["lora"]["r"]=64, alpha=32` (γ = 0.5, the *unstabilised* high-rank LoRA) against the default rsLoRA r = 64 (only the scaling differs). **PEFT vs full adaptation:** `METHOD="full"`, and `METHOD="partial"` with `ATTN_MODE="bidir_top"` (the cheapest of all: only the top blocks get gradients). **Attention:** `ATTN_MODE="causal"` (does unmasking help?). These are for the *analysis*; the 6 required submissions must be different **techniques**, not hyper-parameter variants.
5. **KD:** with the adapters this run (~62M) is a valid teacher for the 36.6M student; set `EXPORT_SOFT_LABELS=True` only if it ends up being your best dev run (the plan uses Dec-02 + LoRMA).
6. Copy `run_log.json` (+ `dev_errors.json`) into the report folder.